# Task 4: Insights and Recommendations

## Business Objective
Synthesize sentiment and thematic analysis into business-actionable insights,
supported by clear visualizations and concrete product recommendations for
three Ethiopian banks.

**Author:** Sosina Ayele

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

print('Libraries imported successfully')

## 1. Load Data

In [ ]:
paths = [
    '../data/review_analysis.csv',
    r'c:\KAIM\fintech-review-analytics\data\review_analysis.csv',
]

df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='utf-8-sig')
        print(f'Loaded from: {path}')
        break

df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(f'Shape: {df.shape}')
print(f'\nReviews per bank:')
print(df['bank'].value_counts().to_string())
df.head()

## 2. Visualization 1 — Sentiment Distribution by Bank

In [ ]:
sentiment_bank = df.groupby(['bank', 'sentiment_label']).size().unstack(fill_value=0)
sentiment_pct = sentiment_bank.div(sentiment_bank.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar — counts
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
bottom = np.zeros(len(sentiment_bank))
for label in ['positive', 'neutral', 'negative']:
    if label in sentiment_bank.columns:
        axes[0].bar(sentiment_bank.index, sentiment_bank[label],
                   bottom=bottom, label=label.capitalize(),
                   color=colors[label])
        bottom += sentiment_bank[label].values
axes[0].set_title('Sentiment Distribution by Bank (Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Bank')
axes[0].set_ylabel('Number of Reviews')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

# Stacked bar — percentage
bottom = np.zeros(len(sentiment_pct))
for label in ['positive', 'neutral', 'negative']:
    if label in sentiment_pct.columns:
        axes[1].bar(sentiment_pct.index, sentiment_pct[label],
                   bottom=bottom, label=label.capitalize(),
                   color=colors[label])
        bottom += sentiment_pct[label].values
axes[1].set_title('Sentiment Distribution by Bank (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Bank')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

plt.suptitle('Bank Sentiment Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_by_bank.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved!')

## 3. Visualization 2 — Rating Distribution per Bank

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
banks = df['bank'].unique()
data_by_bank = [df[df['bank'] == b]['rating'].values for b in banks]
bp = axes[0].boxplot(data_by_bank, labels=[b.split()[-1] for b in banks],
                     patch_artist=True)
colors_box = ['#3498db', '#e67e22', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Rating Distribution (Boxplot)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Star Rating')
axes[0].set_xlabel('Bank')

# Average rating bar
avg_ratings = df.groupby('bank')['rating'].mean().sort_values(ascending=False)
bars = axes[1].bar(avg_ratings.index, avg_ratings.values,
                   color=colors_box)
axes[1].set_title('Average Star Rating by Bank', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(0, 5)
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, avg_ratings.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', fontweight='bold')

plt.suptitle('Rating Analysis by Bank', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved!')

## 4. Visualization 3 — Theme Frequency per Bank

In [ ]:
theme_bank = df.groupby(['bank', 'identified_theme']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, bank in enumerate(df['bank'].unique()):
    bank_themes = df[df['bank'] == bank]['identified_theme'].value_counts()
    axes[i].barh(bank_themes.index[::-1], bank_themes.values[::-1],
                color=sns.color_palette('viridis', len(bank_themes)))
    axes[i].set_title(f'{bank}\nTheme Frequency', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Count')

plt.suptitle('Theme Frequency per Bank', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('theme_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved!')

## 5. Visualization 4 — Sentiment Trend Over Time

In [ ]:
df_time = df.dropna(subset=['date'])
df_time['month'] = df_time['date'].dt.to_period('M')

monthly_sentiment = df_time.groupby(['month', 'bank'])['sentiment_score'].mean().unstack()

plt.figure(figsize=(14, 6))
for bank in monthly_sentiment.columns:
    plt.plot(monthly_sentiment.index.astype(str),
             monthly_sentiment[bank],
             marker='o', markersize=3, linewidth=1.5, label=bank)

plt.title('Sentiment Score Trend Over Time by Bank', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Mean Sentiment Score')
plt.legend()
plt.xticks(rotation=45)
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('sentiment_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved!')

## 6. Visualization 5 — Top Keywords per Bank

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, bank in enumerate(df['bank'].unique()):
    bank_reviews = df[df['bank'] == bank]['review'].astype(str)
    tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1,2))
    matrix = tfidf.fit_transform(bank_reviews)
    scores = pd.DataFrame({
        'keyword': tfidf.get_feature_names_out(),
        'score': matrix.sum(axis=0).A1
    }).sort_values('score', ascending=False).head(10)

    axes[i].barh(scores['keyword'][::-1], scores['score'][::-1],
                color=sns.color_palette('magma', len(scores)))
    axes[i].set_title(f'{bank}\nTop Keywords', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('TF-IDF Score')

plt.suptitle('Top Keywords per Bank (TF-IDF)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved!')

## 7. Insights — Drivers and Pain Points per Bank

In [ ]:
print('=' * 60)
print('INSIGHTS PER BANK')
print('=' * 60)

for bank in df['bank'].unique():
    bank_df = df[df['bank'] == bank]
    pos = bank_df[bank_df['sentiment_label'] == 'positive']
    neg = bank_df[bank_df['sentiment_label'] == 'negative']

    print(f'\n--- {bank} ---')
    print(f'Total reviews: {len(bank_df)}')
    print(f'Average rating: {bank_df["rating"].mean():.2f}')
    print(f'Positive: {len(pos)} ({len(pos)/len(bank_df)*100:.1f}%)')
    print(f'Negative: {len(neg)} ({len(neg)/len(bank_df)*100:.1f}%)')
    print(f'Top themes: {bank_df["identified_theme"].value_counts().head(3).to_dict()}')

## 8. Business Recommendations

### Commercial Bank of Ethiopia
**Satisfaction Drivers:**
1. Fast and reliable money transfer functionality praised in positive reviews
2. Simple and easy-to-use interface appreciated by users

**Pain Points:**
1. Login and OTP issues frequently mentioned in 1-star reviews
2. App crashes and slow loading during peak hours

**Recommendations:**
1. Implement biometric authentication (fingerprint/face ID) to reduce login friction
2. Optimize server capacity during peak banking hours to reduce crashes

---

### Bank of Abyssinia
**Satisfaction Drivers:**
1. Mobile transfer speed appreciated by users
2. Regular app updates noted positively

**Pain Points:**
1. Customer support response time frequently criticized
2. Transaction failures and error messages frustrate users

**Recommendations:**
1. Add in-app live chat support to reduce response time
2. Improve transaction error messages with clear retry options

---

### Dashen Bank
**Satisfaction Drivers:**
1. Modern UI design praised compared to competitors
2. Feature-rich app with good notification system

**Pain Points:**
1. Account access issues (locked accounts, expired sessions)
2. Missing features like balance alerts and scheduled payments

**Recommendations:**
1. Add balance threshold alerts and scheduled payment features
2. Implement automatic session refresh to reduce logout complaints

---

## 9. Ethical Considerations

### Potential Biases:
1. **Negativity Bias:** Users are significantly more likely to leave reviews after a bad experience, skewing the dataset toward negative sentiment
2. **Sampling Bias:** English-language filter may exclude non-English speaking users who may have different experiences
3. **Recency Bias:** Newer reviews may reflect recent app updates rather than long-term quality
4. **Selection Bias:** Only Google Play Store reviews — excludes iOS users who may have different experiences
5. **Volume Imbalance:** Different review counts per bank make direct comparison challenging

In [ ]:
# Final summary table
summary = df.groupby('bank').agg(
    total_reviews=('review', 'count'),
    avg_rating=('rating', 'mean'),
    avg_sentiment=('sentiment_score', 'mean'),
    pct_positive=('sentiment_label', lambda x: (x == 'positive').mean() * 100),
    pct_negative=('sentiment_label', lambda x: (x == 'negative').mean() * 100),
    top_theme=('identified_theme', lambda x: x.value_counts().index[0])
).round(2)

print('=== Final Summary Table ===')
print(summary.to_string())